In [ ]:
import sys
sys.path.append("..")   # add main_folder to path


import geopandas as gpd
import itertools
import pyarrow.dataset as pds
from tqdm import tqdm
from pathlib import Path

from shapely.prepared import prep

import tomllib

from src.genesis.genesis import simulate_tc_genesis
from src.genesis.genesis_utils import read_ibtracs, preprocess_ibtracs


In [ ]:
config_path = "../config.toml"
with open(
    config_path,
    "rb",
) as f:  # Open the file in binary mode
    config_files = tomllib.load(f)

In [ ]:
main_config = config_files["main_params"]
gen_config = config_files["generation"]
intens_config = config_files["intensification"]
n_seeds = gen_config["n_seeds"]

catherina_fit_path = ".." / Path(gen_config["fit_dir"]) / "Catherina_fit.db"
cyclones_dir = Path(gen_config["synthetic_tracks_dir"])

ibtracks_file_path = '..' / Path(gen_config["ibtracs_path"])

data_dir = ".." / Path(main_config["data_dir"])

ne_10m_coastline_zip = ".." / Path(main_config["data_dir"]) / "ne_10m_coastline.zip"
ne_10m_land_zip = ".." /Path(main_config["data_dir"]) / "ne_10m_land.zip"
model_exp_pbar = tqdm(
    list(itertools.product(main_config["models"], main_config["experiments"]))
)
land = gpd.read_file(ne_10m_land_zip).union_all()
land_union = prep(land)


In [ ]:
# Generate synthetic genesis points
usecols = [
    "SID",
    "ISO_TIME",
    "BASIN",
    "LON",
    "LAT",
    "NATURE",
    "TRACK_TYPE",
    "WMO_WIND",
    "WMO_PRES",
]
ibtracs = read_ibtracs(fpath=ibtracks_file_path, usecols=usecols)
ibtracs = preprocess_ibtracs(ibtracs, 32.92)   

genesis_save_dir = '..' / Path(main_config["data_dir"]) / "catherina_ssp585"

#TODO: set displace=True
simulate_tc_genesis(
    ibtracs=ibtracs,
    land_union=land_union,
    resolution=gen_config["resolution"],
    n_seeds=n_seeds,
    start_year=gen_config["start_year"],
    end_year=main_config["end_year"],
    save_dir=genesis_save_dir,
    displace=False,
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def make_genesis_probability_map(
    df: pd.DataFrame,
    *,
    grid_deg: float = 2.0,              # grid size in degrees (e.g., 1.0 or 2.0)
    lat_range: tuple[float, float] = (-60, 60),
    lon_range: tuple[float, float] = (-180, 180),
    basin: str | None = None,           # e.g. "ATL", "EP", "WP", "NI", "SI", "SP"
    years: tuple[int, int] | None = None,
    months: set[int] | list[int] | None = None,
    title: str | None = "Cyclone Genesis Probability",
):
    """
    Plot probability of cyclone genesis on a lat-lon grid from point data.
    Expects df with columns: ['lat','lon', 'basin', 'year', 'month', ...].
    Each row is one genesis point (one genesis per SID/seed/step).

    Returns (fig, ax).
    """
    # -------- filter data (optional) --------
    data = df.copy()
    if basin is not None and "basin" in data.columns:
        data = data[data["basin"] == basin]
    if years is not None and {"year"}.issubset(data.columns):
        y0, y1 = years
        data = data[(data["year"] >= y0) & (data["year"] <= y1)]
    if months is not None and {"month"}.issubset(data.columns):
        months = set(months)
        data = data[data["month"].isin(months)]

    # Guard
    if data.empty:
        raise ValueError("No genesis points after filtering; nothing to plot.")

    # Ensure numeric lat/lon and drop NaNs
    data = data[pd.notna(data["lat"]) & pd.notna(data["lon"])].copy()
    lat = data["lat"].astype(float).to_numpy()
    lon = data["lon"].astype(float).to_numpy()

    # Wrap longitudes to [-180, 180]
    lon = ((lon + 180) % 360) - 180

    # -------- build grid & histogram --------
    lat_min, lat_max = lat_range
    lon_min, lon_max = lon_range
    lat_edges = np.arange(lat_min, lat_max + grid_deg, grid_deg, dtype=float)
    lon_edges = np.arange(lon_min, lon_max + grid_deg, grid_deg, dtype=float)

    # counts per cell
    H, lon_edges_out, lat_edges_out = np.histogram2d(
        lon, lat, bins=[lon_edges, lat_edges]
    )
    # Convert counts to probability (per cell)
    total = H.sum()
    if total <= 0:
        raise ValueError("Histogram is empty; check filters / ranges.")
    P = H / total  # probability mass per cell

    # For pcolormesh, build cell centers (optional; edges are fine)
    lon_centers = 0.5 * (lon_edges_out[:-1] + lon_edges_out[1:])
    lat_centers = 0.5 * (lat_edges_out[:-1] + lat_edges_out[1:])

    # -------- plot --------
    # Try Cartopy for coastlines; fall back to plain Matplotlib
    try:
        import cartopy.crs as ccrs
        import cartopy.feature as cfeature
        proj = ccrs.PlateCarree()
        fig = plt.figure(figsize=(10, 5))
        ax = plt.axes(projection=proj)
        # pcolormesh expects edges in PlateCarree
        mesh = ax.pcolormesh(lon_edges_out, lat_edges_out, P.T, transform=proj)
        ax.coastlines(linewidth=0.8)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4)
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=proj)
        cb = plt.colorbar(mesh, ax=ax, shrink=0.8)
        cb.set_label("Genesis probability per grid cell")
        ax.set_title(title or "Cyclone Genesis Probability")
        # gridlines (optional)
        gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
        gl.right_labels = False
        gl.top_labels = False
    except Exception:
        # Plain Matplotlib fallback (no map projection)
        fig, ax = plt.subplots(figsize=(10, 5))
        extent = [lon_edges_out.min(), lon_edges_out.max(), lat_edges_out.min(), lat_edges_out.max()]
        im = ax.imshow(
            P.T,
            origin="lower",
            extent=extent,
            aspect="auto",
        )
        cb = plt.colorbar(im, ax=ax, shrink=0.8)
        cb.set_label("Genesis probability per grid cell")
        ax.set_xlim(lon_min, lon_max)
        ax.set_ylim(lat_min, lat_max)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.set_title(title or "Cyclone Genesis Probability")
        ax.grid(alpha=0.3)

    return fig, ax

In [ ]:
genesis_ds = pds.dataset(
    '..' / Path(main_config["data_dir"]) / "catherina_historical" / "genesis",
    format="parquet",
    partitioning="hive",
)
df = genesis_ds.to_table().to_pandas()
make_genesis_probability_map(df)

In [ ]:
main_config = config_files["main_params"]
gen_config = config_files["generation"]
data_dir = ".." / Path(main_config["input_data_dir"])
ibtracksf_folder = data_dir / gen_config["ibtracs_path"]

In [ ]:
# Generate synthetic genesis points
# usecols = [
#     "SID",
#     "ISO_TIME",
#     "BASIN",
#     "LON",
#     "LAT",
#     "NATURE",
#     "TRACK_TYPE",
#     "WMO_WIND",
#     "WMO_PRES",
# ]
ibtracs = read_ibtracs(fpath=ibtracksf_folder, signed_coords=True)
ibtracs_pp = preprocess_ibtracs(ibtracs, 32.92)   

In [ ]:
ibtracs.drop_duplicates('SID', keep='first')

In [ ]:
make_genesis_probability_map(ibtracs.drop_duplicates('SID').rename(columns={'LAT':'lat','LON':'lon'}))